# Módulo 5 — Análise Exploratória, Indicadores e Fatores Associados

In [5]:
## Bibliotecas principais do Módulo 4
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import unicodedata

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


In [6]:
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

ARQUIVO_BRUTO = RAIZ / "dados_brutos" / "acidentes2025.csv"
ARQUIVO_BASE_ANALITICA = RAIZ / "dados_tratados" / "base_analitica_prf_2025.csv"
ARQUIVO_BASE_MODELAVEL = RAIZ / "dados_tratados" / "base_modelavel_prf_2025.csv"
ARQUIVO_DICIONARIO = RAIZ / "dados_tratados" / "dicionario_variaveis_modulo4.csv"
ARQUIVO_DECISOES = RAIZ / "logs" / "decisoes_tratamento_modulo4.md"
ARQUIVO_README = RAIZ / "docs" / "README.md"

SEPARADOR = ";"
ENCODING_ENTRADA = "latin1"
ENCODING_SAIDA = "utf-8-sig"

In [9]:
df = pd.read_csv(ARQUIVO_BASE_ANALITICA,
                    sep=SEPARADOR,
                    encoding=ENCODING_SAIDA)
df["acidente_fatal"] = (df["mortos"] >= 1).astype(int)

total     = len(df)
fatais    = df["acidente_fatal"].sum()
pct_fatal = fatais / total
mortos    = df["mortos"].sum()

In [13]:
# Tipos de dados e memória utilizada
df.info(memory_usage="deep")

resumo_tipos = (
 df.dtypes.astype(str)
 .value_counts()
 .rename_axis("tipo")
 .reset_index(name="qtd_colunas")
)
display(resumo_tipos)

<class 'pandas.DataFrame'>
RangeIndex: 72529 entries, 0 to 72528
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      72529 non-null  int64  
 1   data_inversa            72529 non-null  str    
 2   dia_semana              72529 non-null  str    
 3   horario                 72529 non-null  str    
 4   uf                      72529 non-null  str    
 5   br                      72529 non-null  int64  
 6   km                      72529 non-null  float64
 7   municipio               72529 non-null  str    
 8   causa_acidente          72529 non-null  str    
 9   tipo_acidente           72529 non-null  str    
 10  classificacao_acidente  72529 non-null  str    
 11  fase_dia                72529 non-null  str    
 12  sentido_via             72529 non-null  str    
 13  condicao_metereologica  72529 non-null  str    
 14  tipo_pista              72529 non-null  str    
 

,tipo,qtd_colunas
0,str,23
1,int64,20
2,float64,1


In [12]:
df.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano,mes,trimestre,dia_semana_num,fim_de_semana,hora,turno,faixa_horaria,acidente_fatal,total_vitimas,acidente_grave,indice_gravidade,br_formatada,chave_localidade
0,652493,2025-01-01,QUARTA-FEIRA,06:20:00,SP,116,225.0,GUARULHOS,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,TOMBAMENTO,COM VÍTIMAS FERIDAS,PLENO DIA,DECRESCENTE,CÉU CLARO,MÚLTIPLA,"['RETA', 'DECLIVE']",SIM,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP,2025,1,1,2,0,6,MANHA,06H-08H,0,1,0,1,BR-116,SP_GUARULHOS_BR-116
1,652519,2025-01-01,QUARTA-FEIRA,07:50:00,CE,116,546.2,PENAFORTE,PISTA ESBURACADA,COLISÃO FRONTAL,COM VÍTIMAS FATAIS,PLENO DIA,CRESCENTE,CÉU CLARO,SIMPLES,['RETA'],NÃO,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,2025,1,1,2,0,7,MANHA,06H-08H,1,2,1,4,BR-116,CE_PENAFORTE_BR-116
2,652522,2025-01-01,QUARTA-FEIRA,08:45:00,PR,369,88.2,CORNELIO PROCOPIO,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO TRASEIRA,COM VÍTIMAS FERIDAS,PLENO DIA,CRESCENTE,SOL,DUPLA,"['RETA', 'ACLIVE']",SIM,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR,2025,1,1,2,0,8,MANHA,06H-08H,0,3,0,3,BR-369,PR_CORNELIO PROCOPIO_BR-369
3,652544,2025-01-01,QUARTA-FEIRA,11:00:00,PR,116,74.0,CAMPINA GRANDE DO SUL,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,SAÍDA DE LEITO CARROÇÁVEL,COM VÍTIMAS FERIDAS,PLENO DIA,CRESCENTE,CÉU CLARO,DUPLA,['RETA'],NÃO,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR,2025,1,1,2,0,11,MANHA,09H-11H,0,1,0,1,BR-116,PR_CAMPINA GRANDE DO SUL_BR-116
4,652549,2025-01-01,QUARTA-FEIRA,09:30:00,MG,251,471.0,FRANCISCO SA,VELOCIDADE INCOMPATÍVEL,COLISÃO FRONTAL,COM VÍTIMAS FERIDAS,PLENO DIA,DECRESCENTE,CHUVA,SIMPLES,"['CURVA', 'DECLIVE']",NÃO,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG,2025,1,1,2,0,9,MANHA,09H-11H,0,2,1,3,BR-251,MG_FRANCISCO SA_BR-251


In [10]:
def taxa_fatal_por_categoria(base, coluna, min_registros=30):
    tab = base.groupby(coluna).agg(
    qtd_acidentes=("acidente_fatal","size"),
    qtd_fatais=("acidente_fatal","sum"),
    taxa_fatal=("acidente_fatal","mean")
    ).reset_index()
    tab = tab[tab["qtd_acidentes"] >= min_registros]
    return tab.sort_values("taxa_fatal", ascending=False)

display(taxa_fatal_por_categoria(df, "tipo_acidente",min_registros=30).head(10))


,tipo_acidente,qtd_acidentes,qtd_fatais,taxa_fatal
1,ATROPELAMENTO DE PEDESTRE,3057,902,0.295061
4,COLISÃO FRONTAL,4739,1396,0.294577
6,COLISÃO LATERAL SENTIDO OPOSTO,2152,212,0.098513
11,EVENTOS ATÍPICOS,287,23,0.080139
0,ATROPELAMENTO DE ANIMAL,1133,68,0.060018
14,SAÍDA DE LEITO CARROÇÁVEL,10209,605,0.059261
3,COLISÃO COM OBJETO,5109,297,0.058133
2,CAPOTAMENTO,1373,63,0.045885
7,COLISÃO TRANSVERSAL,9306,427,0.045884
8,COLISÃO TRASEIRA,14360,619,0.043106


In [ ]:
def indicadores_por_categoria(base, coluna, min_registros=30):
    tab = base.groupby(coluna).agg(
    qtd_acidentes=("acidente_fatal","size"),
    qtd_fatais=("acidente_fatal","sum"),
    taxa_fatal=("acidente_fatal","mean")
    ).reset_index()
    tab = tab[tab["qtd_acidentes"] >= min_registros]
    return tab.sort_values("taxa_fatal", ascending=False)

display(taxa_fatal_por_categoria(df, "tipo_acidente",min_registros=30).head(10))

In [11]:
display(taxa_fatal_por_categoria(df, "causa_acidente",min_registros=30).head(10))

,causa_acidente,qtd_acidentes,qtd_fatais,taxa_fatal
60,SUICÍDIO (PRESUMIDO),190,106,0.557895
45,PEDESTRE ANDAVA NA PISTA,606,250,0.412541
27,ENTRADA INOPINADA DO PEDESTRE,667,203,0.304348
63,TRANSITAR NA CONTRAMÃO,2475,736,0.297374
46,PEDESTRE CRUZAVA A PISTA FORA DA FAIXA,525,130,0.247619
30,FALTA DE ACOSTAMENTO,60,11,0.183333
44,PEDESTRE - INGESTÃO DE ÁLCOOL/ SUBSTÂNCIAS PSI...,150,27,0.180000
66,ULTRAPASSAGEM INDEVIDA,1770,302,0.170621
65,TRANSTORNOS MENTAIS (EXCETO SUICIDIO),59,10,0.169492
68,ÁREA URBANA SEM A PRESENÇA DE LOCAL APROPRIADO...,81,12,0.148148
